# Naive IQ-Learn on PointMaze Medium

In [1]:
import random
import copy
import torch
import pickle
import os
import numpy as np
import matplotlib.pyplot as plt

from collections import defaultdict

from causal_gym import PointMazePCH
from causal_rl.algo.imitation.imitate import *
from causal_rl.algo.imitation.finetune import *
from causal_rl.algo.imitation.gail.core_net import ContinuousActor
from causal_rl.algo.imitation.gail.causal_gail import *
from causal_rl.algo.imitation.iqlearn.core_net import IQLearnQNetwork
from causal_rl.algo.imitation.iqlearn.causal_iqlearn import (
    IQLearnReplayBuffer, iq_init_expert_buffer,
    rollout_iqlearn_episode, iqlearn_update_critic, iqlearn_update_actor,
    soft_update, evaluate_iqlearn_policy,
)

<frozen importlib._bootstrap>:241: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.
/home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '7'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
num_steps = 1000
seed = 0
lookback = 1
hidden_dims = {'H'}

random.seed(seed)
torch.manual_seed(seed)

In [4]:
# for training: regular W, L hidden
train_env = PointMazePCH(num_steps=num_steps, expert_mode=True, custom_hidden=hidden_dims, seed=seed)

# for eval: corrupted W, L hidden
eval_env = PointMazePCH(num_steps=num_steps, expert_mode=False, seed=seed)

## Causal Graph Analysis

In [5]:
# to save time; conceptually the same
small_steps = lookback + 1
small_env = PointMazePCH(num_steps=small_steps, seed=seed)
G = parse_graph(small_env.get_graph)
X_small = {f'X{t}' for t in range(small_steps)}
Y = f'Y{small_steps}'

X = {f'X{t}' for t in range(num_steps)}
obs_prefix = train_env.env.observed_unobserved_vars[0]

In [6]:
naive_Z_sets = {}
for Xi in X:
    i = int(Xi[1:])
    cond = set()

    for j in range(i+1):
        cond.update({f'{o}{j}' for o in list(set(obs_prefix) - {'X'})})

    for j in range(i):
        cond.add(f'X{j}')
    naive_Z_sets[Xi] = cond

naive_Z_sets['X1']

{'D0', 'D1', 'V0', 'V1', 'X0'}

## Expert Trajectories

In [7]:
with open('/home/et2842/causal/expert_traj_pointmed.pkl', 'rb') as f:
    records = pickle.load(f)

print(f'loaded {len(records)} trajectories')

loaded 104826 trajectories


In [8]:
dims = {
    'V': 1,
    # 'H': 1,
    'D': 1,
    'X': 2
}

In [9]:
sample_obs = records[0]['obs']

# Trim Z-sets to the lookback window
naive_Z_trim = trim_Z_sets(naive_Z_sets, lookback=lookback)

# Build windowed encoders that depend on relative lags
naive_encode, naive_z_dim, naive_slots = build_windowed_z_encoder(
    naive_Z_trim,
    dims=dims,
    lookback=lookback,
)

encode = naive_encode
z_dim = naive_z_dim
Z_trim = naive_Z_trim
naive_z_dim

6

## Hyperparameters

In [10]:
# Shared SAC hyperparameters
total_timesteps = 2_000_000
batch_size = 256
gamma = 0.99
tau = 0.005
actor_lr = 3e-4
critic_lr = 3e-4
alpha_lr = 3e-4
hidden_dim = 256
buffer_capacity = 1_000_000
expert_capacity_ratio = 0.5
start_steps = 5_000
log_every = 50
eval_episodes = 10
max_grad_norm = 1.0
utd_ratio = 0.25  # update-to-data ratio: 1 gradient update per 4 env steps

# Actor architecture (match GAIL)
num_blocks_actor = 3
dropout_actor = 0.05
layernorm_actor = True

# IQ-Learn specific
num_v_samples = 16

# Environment action space
action_dim = train_env.env.action_space.shape[0]
action_low = float(train_env.env.action_space.low.min())
action_high = float(train_env.env.action_space.high.max())
target_entropy = -float(action_dim)

## Network Initialization

In [11]:
actor = ContinuousActor(
    num_inputs=z_dim, num_outputs=action_dim,
    hidden_size=hidden_dim, std=0.0,
    action_low=action_low, action_high=action_high,
    num_blocks=num_blocks_actor, dropout=dropout_actor, layernorm=layernorm_actor,
).to(device)

q1 = IQLearnQNetwork(z_dim, action_dim, hidden_dim,
                      num_blocks=num_blocks_actor, dropout=dropout_actor,
                      layernorm=layernorm_actor).to(device)
q2 = IQLearnQNetwork(z_dim, action_dim, hidden_dim,
                      num_blocks=num_blocks_actor, dropout=dropout_actor,
                      layernorm=layernorm_actor).to(device)
tq1 = copy.deepcopy(q1)
tq2 = copy.deepcopy(q2)
for p in tq1.parameters(): p.requires_grad = False
for p in tq2.parameters(): p.requires_grad = False

actor_optim = torch.optim.Adam(actor.parameters(), lr=actor_lr)
q1_optim = torch.optim.Adam(q1.parameters(), lr=critic_lr)
q2_optim = torch.optim.Adam(q2.parameters(), lr=critic_lr)

# Cosine LR schedule for critics
estimated_total_updates = int(total_timesteps * utd_ratio)
q1_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(q1_optim, T_max=estimated_total_updates)
q2_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(q2_optim, T_max=estimated_total_updates)

# Automatic entropy tuning
log_alpha = torch.zeros(1, requires_grad=True, device=device)
alpha_optim = torch.optim.Adam([log_alpha], lr=alpha_lr)

buffer = IQLearnReplayBuffer(buffer_capacity, expert_capacity_ratio)
iq_init_expert_buffer(records, encode, buffer, device)

Expert buffer: 104826 transitions from 1000 episodes


## Training

In [12]:
best_eval = -float('inf')
best_state_dict = copy.deepcopy(actor.state_dict())

ts = 0
ep = 0
logs = []

while ts < total_timesteps:
    ep_data = rollout_iqlearn_episode(
        train_env, actor, buffer, encode,
        num_steps, device, deterministic=False, seed=seed + 30000 + ep
    )
    ts += ep_data['episode_length']
    ep += 1

    if ts > start_steps and len(buffer.policy_buffer) >= batch_size // 2:
        n_updates = max(1, int(ep_data['episode_length'] * utd_ratio))
        for _ in range(n_updates):
            alpha_val = log_alpha.exp().item()
            iqlearn_update_critic(
                q1, q2, tq1, tq2, actor, alpha_val, buffer,
                batch_size, gamma, q1_optim, q2_optim,
                device, num_v_samples, max_grad_norm,
            )
            iqlearn_update_actor(
                actor, q1, q2, log_alpha, target_entropy,
                actor_optim, alpha_optim,
                buffer, batch_size, device, max_grad_norm,
            )
            soft_update(q1, tq1, tau)
            soft_update(q2, tq2, tau)

            q1_scheduler.step()
            q2_scheduler.step()

            # Alpha clamping (IQ-Learn stability fix)
            with torch.no_grad():
                log_alpha.clamp_(min=np.log(0.001), max=np.log(0.1))

    if ep % log_every == 0:
        eval_ret = evaluate_iqlearn_policy(
            train_env, actor, encode, num_steps, device, eval_episodes, seed=42
        )
        logs.append({
            'episode': ep, 'timesteps': ts,
            'eval_return': eval_ret, 'train_return': ep_data['episode_return'],
            'alpha': log_alpha.exp().item(),
        })
        print(
            f"[Naive IQ-Learn ep {ep}] "
            f"ts={ts}, eval={eval_ret:.2f}, "
            f"train={ep_data['episode_return']:.2f}, "
            f"alpha={log_alpha.exp().item():.4f}"
        )

        # Best checkpoint tracking
        if eval_ret > best_eval:
            best_eval = eval_ret
            best_state_dict = copy.deepcopy(actor.state_dict())

# Restore best
actor.load_state_dict(best_state_dict)
print(f"Restored best checkpoint with eval={best_eval:.2f}")

[Naive IQ-Learn ep 50] ts=11917, eval=-5.48, train=-5.16, alpha=0.0952


[Naive IQ-Learn ep 100] ts=18328, eval=-5.51, train=1.00, alpha=0.1000


[Naive IQ-Learn ep 150] ts=25047, eval=-5.43, train=-9.01, alpha=0.1000


[Naive IQ-Learn ep 200] ts=31270, eval=-5.41, train=-5.06, alpha=0.1000


[Naive IQ-Learn ep 250] ts=42513, eval=-5.15, train=-13.01, alpha=0.1000


[Naive IQ-Learn ep 300] ts=54634, eval=-50.27, train=-16.01, alpha=0.1000


## Evaluation

In [ ]:
naive_iqlearn_policy = make_gail_policy(actor, encode, device=device, deterministic=True)
naive_iqlearn_policies = make_shared_policy_dict(naive_iqlearn_policy)

In [ ]:
num_eval_eps = 100
naive_iqlearn_returns = collect_imitator_trajectories(
    env=eval_env,
    policies=naive_iqlearn_policies,
    num_episodes=num_eval_eps,
    max_steps=num_steps,
    hidden_dims=hidden_dims,
    show_progress=True,
    seed=seed + 90210,
)

len(naive_iqlearn_returns)

In [ ]:
naive_iqlearn_episode_rewards = defaultdict(float)
for rec in naive_iqlearn_returns:
    ep = rec['episode']
    naive_iqlearn_episode_rewards[ep] += float(rec['reward'])

naive_iqlearn_rewards = [naive_iqlearn_episode_rewards[e] for e in range(num_eval_eps)]
sum(naive_iqlearn_rewards) / num_eval_eps

## Save Model

In [ ]:
SAVE_DIR = '/home/et2842/causal/causalrl/models'
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_PATH = os.path.join(SAVE_DIR, 'niqlearn_pointmed.pt')

ckpt = {
    'state_dict': actor.state_dict(),
    'z_dim': naive_z_dim,
    'action_dim': action_dim,
    'hidden_size_actor': hidden_dim,
    'num_blocks_actor': num_blocks_actor,
    'dropout_actor': dropout_actor,
    'layernorm_actor': layernorm_actor,
    'final_tanh': True,
    'action_bounds_low': eval_env.env.action_space.low,
    'action_bounds_high': eval_env.env.action_space.high,
    'Z_sets': naive_Z_trim,
    'dims': dims,
    'lookback': lookback,
}

torch.save(ckpt, MODEL_PATH)
print(f'Saved to: {MODEL_PATH}')